# Feature extraction sanity check

Exercises the real pipeline path: `preprocess_subjects` (interpolate + filter + window)
then `extract_features` / `extract_all_features` on `HeartbeatDataProcessor`.

An earlier version of this notebook called `scipy.stats.hmean` and a bare `df.agg([...])`
on a raw, un-interpolated subject. That is not the pipeline: `scipy.stats.hmean` returns
NaN on any NaN or negative input, so it showed NaNs (heart rate is ~91% NaN raw; the
accelerometer axes are signed) that the pipeline's own harmonic mean (signed, NaN-dropping,
in `data_preprocessing.py`) does not produce. Use this notebook to check what the pipeline
actually computes.

In [ ]:
from data_preprocessing import HeartbeatDataProcessor

folder_path = '../data/PAMAP2_Dataset/Protocol/'
filtered_df_path = './'

processed_data = HeartbeatDataProcessor(folder_path, filtered_df_path)
processed_data.preprocess_subjects([101])

## One window

`extract_features` returns one row indexed `<channel>_<feature>`, over the channels the
pipeline keeps (heart rate plus the 27 motion axes; no timestamp, activity_id, or
temperature). Below: the row size, its NaN count, every harmonic-mean value (the statistic
the old notebook flagged), and one channel's full set of statistics.

In [ ]:
# first non-empty window of subject 101
window = next(w for w in processed_data.subject_segment_dict[101] if not w.empty)

row = processed_data.extract_features(window)
print('feature count:', len(row))
print('NaNs in this row:', int(row.isna().sum()))

# every harmonic-mean value: finite here, unlike scipy.stats.hmean on raw data
row[[c for c in row.index if c.endswith('_hmean')]]

In [ ]:
# all statistics for one example channel
channel = 'hand_acc16_x'
row[[c for c in row.index if c.startswith(channel + '_')]]

## Full matrix

The NaN check: the harmonic-mean columns, and the matrix as a whole, should be NaN-free on
the pipeline path.

In [ ]:
features_df = processed_data.extract_all_features()
print('shape:', features_df.shape)
print('total NaNs:', int(features_df.isna().sum().sum()))

hmean_cols = [c for c in features_df.columns if c.endswith('_hmean')]
print('hmean columns:', len(hmean_cols), '| NaNs in them:', int(features_df[hmean_cols].isna().sum().sum()))
features_df.head(3)